# Principal Component Analysis (PCA)
Accompanies **Chapter 22: Principal Component Analysis (PCA)**.

PCA finds orthogonal direction vectors $\mathbf{u}_j$ that maximize projected sample variance:
$$\max_{\mathbf{u}} \mathbf{u}^T \mathbf{\Sigma} \mathbf{u} \quad \text{s.t.} \quad \mathbf{u}^T \mathbf{u} = 1$$
Forming the Lagrangian $\mathcal{L} = \mathbf{u}^T \mathbf{\Sigma} \mathbf{u} - \lambda (\mathbf{u}^T \mathbf{u} - 1)$ leads to the fundamental eigenvalue equation:
$$\mathbf{\Sigma} \mathbf{u}_j = \lambda_j \mathbf{u}_j$$
The variance along direction $\mathbf{u}_j$ is exactly eigenvalue $\lambda_j$.

In this notebook, we implement PCA via sample covariance eigendecomposition, construct a Scree Plot, and project high-dimensional data into 2D.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_wine

data = load_wine()
X = data.data
y = data.target

# Standardize to zero mean and unit variance (mandatory for variance comparisons)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
n_samples, n_features = X_scaled.shape

### Eigendecomposition of the Sample Covariance Matrix
$$\mathbf{\Sigma} = \frac{1}{n-1} \tilde{\mathbf{X}}^T \tilde{\mathbf{X}}$$
Eigenvalues $\lambda_1 \ge \lambda_2 \ge \dots \ge \lambda_p$ give the variance explained by each principal component.

In [ ]:
cov_matrix = (X_scaled.T @ X_scaled) / (n_samples - 1)
eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)

# Sort descending
sort_idx = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[sort_idx]
eigenvectors = eigenvectors[:, sort_idx]

total_variance = np.sum(eigenvalues)
evr_custom = eigenvalues / total_variance

# Compare against Scikit-Learn SVD implementation
sk_pca = PCA().fit(X_scaled)
assert np.allclose(evr_custom, sk_pca.explained_variance_ratio_)
print("Variance ratios match Scikit-Learn exactly.")

### The Scree Plot & Cumulative Explained Variance

In [ ]:
cum_evr = np.cumsum(sk_pca.explained_variance_ratio_)

fig, ax1 = plt.subplots(figsize=(9, 4.8))
ax1.bar(range(1, 14), sk_pca.explained_variance_ratio_, alpha=0.5, color='steelblue', label='Individual Variance')
ax1.set_xlabel('Principal Component Index ($j$)', fontsize=11)
ax1.set_ylabel('Variance Ratio', color='steelblue', fontsize=11)

ax2 = ax1.twinx()
ax2.plot(range(1, 14), cum_evr, color='crimson', marker='o', linewidth=2, label='Cumulative Variance')
ax2.axhline(0.85, color='gray', linestyle='--', linewidth=1.5, label='85% Threshold')
ax2.set_ylabel('Cumulative Ratio', color='crimson', fontsize=11)

plt.title('PCA Scree Plot: Locating the Variance Elbow', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3)
plt.show()

k_retained = np.argmax(cum_evr >= 0.85) + 1
print(f"Components needed to retain >= 85% variance: {k_retained} (out of {n_features})")

### 2D Subspace Projection

In [ ]:
# Score matrix: Z_k = X U_k
Z_2d = X_scaled @ eigenvectors[:, :2]

plt.figure(figsize=(8, 5.5))
scatter = plt.scatter(Z_2d[:, 0], Z_2d[:, 1], c=y, cmap='viridis', edgecolors='k', s=50, alpha=0.8)
plt.xlabel(f'PC 1 ({evr_custom[0]:.1%} variance)', fontsize=11)
plt.ylabel(f'PC 2 ({evr_custom[1]:.1%} variance)', fontsize=11)
plt.title('Wine Dataset Projected onto Top 2 Principal Components', fontsize=12, fontweight='bold')
plt.colorbar(scatter, label='Cultivar Class')
plt.grid(True, alpha=0.3)
plt.show()